<div style="display: flex; align-items: center;">
  <img src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/SemanticKernelLogo.png" width="40px" style="margin-right: 10px;">
  <span style="font-size: 1.5em; font-weight: bold;">3c - Workshop (AI Extensions) - Open Source Decision Intelligence</span>
</div>

Decision Intelligence applied in this module:  
* Listing key factors to consider when making a sound decision  
* Using OSS (open-source) reasoning models to optimize the decision approach 
* Decision Scenario: Use a decision framework (Ben Franklin's Pro & Con List) to create a decision plan  
* Improving Decision Intelligence process by explicitly providing decision frameworks and additional context 

A recommended enterprise pattern is to scale Articial Intelligence strategy with three key pillars:
* **Commercial AI** (OpenAI and other proprietary Generative AI providers)
* **Open Source (OpenWeight) AI** (open-source AI providers)
* **Vendor and Partner AI** (i.e. company HR Software, contract software) 

> 📝 Note: There is a technical difference between Open Source and Open Weight models. Open Weight models generally only provide you the weights & biases file(s) with a friendly open source license. Open Source models include the pre/post training data, training scripts and sometimes the training checkpoints. Those three additions allow the models to be fully retrained by anyone. More info can be found here: https://opensource.org/ai/open-weights. For this workshop, we will use the term open source more loosely. A great majority of the cases most organizations just want two things: the weights & biases file(s) and a friendly licnese. Certain organizations certainly require full model assets for risk management. While the full open source model assets are nice to have, they offer minimal value in most cases. 

These three pillars (listed above) strategically form AI capability and capacity in what I like to refer to as the **"Generative AI Brain"**. This is illustrated below with sample providers. Most organizations scale their AI investments with Commercial AI providers, such as: OpenAI, Anthropic or Google. However, some organizations prefer to have more governance controls over their AI and prefer open-source alternatives. For hobby consumers, open source translates to no credit card requirements for APIs and very high privacy. 
Note that there are some commercial providers like Meta or Cohere that provide open weight models. Finally in the third tier, an organization may have an existing relationship with a vendor/partner that has their own AI integrations. For example, Adobe offers enterprise graphic design/contract software and leveraging their built-in Generative AI capabilities rather building their own can make a lot of sense. 

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/AIBrainPillars.png">

Most developer AI frameworks work across all the pillars mentioned above. It allows all types of models (commercial or proprietary) and almost any APIs to be orchestrated to faciliate enterprise Decision Intelligence. This is a much deeper subject and for simplicity this module highlights how to use local GenAI models (i.e. LLMs) with Microsoft AI for decision-making. Models from the OpenAI OSS family will be selected to run locally using LMStudio as a local REST endpoint that will interface with Microsoft AI orchestration.  

Open Source (OSS) models vary dramatically in size. They can have a small number of parameters/activated layers and perform great locally on your mobile device! They can also have a huge number parameters that rivals commercial AI LLMs. 

In the cases where OSS models have a small number of parameters, they are are considered SLMs (Small Language Models) with parameters generally below the 30 Billion parameter threshhold. This allows most of these models to run comfortably on commodity hardware available to personal users. This doesn't mean these models are not capable of performing well. While these models certainly may lack the general knowledge breadth of frontier AI Large Language Models, SLMs make up for it by providing very specialized logic, math and reasoning capabilities. For example, an SLM only trained for a specific language (English) and a certain domain (legal industry) can include only the relevant training information. Therefore, it can be offered as a much more compact model with many less parameters. 

In the cases where OSS models have a large number of parameters, they can perform just as well as commercial models! OSS models with large parameters require enterprise commmerical hardware to execute at scale. Below is an image from an independent AI benchmarking site ArtificialAnalysis.ai across proprietary and OSS models. Notice that almost half of the Top 30 performing models are open weight models (blue): 

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/refs/heads/main/Images/ArtificialAnalysis-ModelBenchmark-202604.png">  

For this workshop, we will be working primarily with SLM family of OSS models, because not everyone has H100 or B200 Nvidia GPUs to deploy huge parameter models. However, as you will see these smaller models are quite good. As of April 2026, OpenAI's gpt-oss-20 or Gemma 4 open-source reasoning models performs about 2 generations back of frontier LLM performance. Keep in mind these are general 20B & 26B parameter OSS models. This model can be fine-tuned or mid-trained and can perform much better. 

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/refs/heads/main/Images/LMStats-Benchmark-202604.png">  

----
### Step 1 - Get Started with LMStudio and Local Open Source AI Models 

Steps to get started:
* Download & install the latest LMStudio version: https://lmstudio.ai/ (Windows, Mac or Linux) 
* Run the LMStudio studio application.
* In the LMStudio application, download the **gpt-oss-20b** GGUF (MLX is on macOS) file. Search for **"OpenAI"** in the "Discover" section of LMStudio. A variety of OpenAI options that are official and unofficial from hobbyists will appear. Typically, selecting the official model with the most downloads will provide the best results. You can select different quantizations of the model, to optimize the performance  
* In the experiment below, the 20B parameter model is being used. LMStudio will inspect your hardware and let you know which quantized version of the model(s) is optimal for your hardware. Note: Even computers with small graphics cards can run these models well locally. Furthermore, laptops such as the Macbook Pro with Neural Engine can run LMStudio local models as well. You just need memory. 
* Start the LMStudio Server with the Phi-4 model loaded. This will start a local REST endpoint with a URI similar to http://10.0.0.18:1234/v1
* The LMStudio local server does not have default security, you can simply check by navigating to this link in any browser to check if a model is loaded: **http://10.0.0.18:1234/v1/models** in the web browser. 

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/LMStudioServer.png"> 

----
### Step 2 - Initialize ChatClient using OpenAI libraries

Execute the next cell to:
* Use the Configuration Builder to use the local LMStudio Server  
* Use the API configuration to build the Semantic Kernel orchestrator  
* Notice there is no security being passed in and it is simply a URL

In [1]:
// Install the OpenAI NuGet Package
#r "nuget: Microsoft.Extensions.DependencyInjection, 10.0.6"
#r "nuget: OpenAI, 2.10.0"

using OpenAI;
using OpenAI.Chat;
using OpenAI.Responses;
using System.ClientModel; // used for APiKeyCredential

// Ollama (you can use this if wanting to use Ollama locally with MEAI extensions)
// IChatClient chatClient = new OllamaApiClient(new Uri("http://127.0.0.1:11434"), "gpt-oss:20b");

var apiKey = "not_needed_for_lmstudio_ollama_etc";
var localUrl = "http://10.0.0.61:1234/v1/"; // this can be localhost or an IP address
var localModelName = "google/gemma-4-26b-a4b"; //"openai/gpt-oss-20b";

var apiCredentials = new ApiKeyCredential(apiKey);
var openAIClientOptions = new OpenAIClientOptions
{
    Endpoint = new Uri(localUrl)
};

// Create a local AI client 
var localAIClient = new OpenAIClient(apiCredentials, openAIClientOptions);

// Create a chat client for the specified local model
var localAIChatClient = localAIClient.GetChatClient(localModelName);

// #pragma warning disable
// var localAIChatClientTwo = localAIClient.GetChatClient(localModelName);
// #pragma warning restore

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.Extensions.DependencyInjection, 10.0.6 OpenAI, 2.10.0

---
### Step 3 - Open Source AI with Decision Intelligence 

Semantic Kernel allows one to interact with any API service that adheres to the OpenAI specification. Notice the method to add LMStudio capability was simply enabled via the **AddOpenAIChatCompletion** method above. 

Execute the cell below about decision factors for a investment property. Note:
* OpenAI Prompt Execution Settings are the same in LMStudio as they are for OpenAI and Azure OpenAI
* OSS models have specific model cards identifying best practices 
* OSS reasoning models will output their reasoning tokens inside the <think> XML tags
* Passing in arguents works the same way in Semantic Kernel as other models 
* Streaming works as well in Semantic Kernel for supported services 

> 📝 **Note:** It is important to note that different model providers (commercial and open-source) have different recommended general settings that have been tested. While different types of AI systems will require optimizing these settings, recommended settings are a good starting point.

In [2]:
// Define the system prompt for the Decision Intelligence assistant
var systemDecisionPrompt = """
You are a Decision Intelligence assistant. 
Assist the user in exploring options, reasoning through decisions, problem-solving, and applying systems thinking to various scenarios. 
Provide structured, logical, and comprehensive advice.

Output Formatting Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower. 
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";

// Create a Decision Intelligence prompt on the topic of purchasing a secondary home as an investment property
// Provide detailed decision-making criteria for evaluating the investment decision
var simpleDecisionPrompt = """
You are considering purchasing a secondary home as an investment property. 

What key factors should you evaluate to ensure a sound investment decision, including financial, 
market, and property-specific considerations? 
Outline the critical steps and criteria for assessing location, potential rental income, 
financing options, long-term property value, and associated risks. 
""";

// Create a new chat history object with proper system and user message roles
var chatMessages = new List<ChatMessage>();

// Add system and user messages to the chat history
var systemMessage = new SystemChatMessage(systemDecisionPrompt);
var userMessage = new UserChatMessage(simpleDecisionPrompt);
chatMessages.Add(systemMessage);
chatMessages.Add(userMessage);
    
#pragma warning disable OPENAI001

// Execute the chat messages against the AI model
var chatHistoryResponse = await localAIChatClient.CompleteChatAsync(chatMessages);
var chatHistoryResponseText = chatHistoryResponse.Value.Content[0].Text;

// Display the response string as Markdown
chatHistoryResponseText.DisplayAs("text/markdown");




// // Render Markdown or stream the chat completion response
// var renderMarkdown = true;
// if (renderMarkdown)
// {
//     // Get the chat completion response as a single content object
//     var chatResponse = await chatCompletionService.GetChatMessageContentAsync(chatHistory, openAIPromptExecutionSettings);
//     var chatResponseString = chatResponse.Content;

//     string endTag = "</think>";
//     // Split into at most 2 chunks; the delimiter itself is discarded.
//     string[] parts = chatResponseString.Split(endTag, 2, StringSplitOptions.None);

//     // Defensive checks
//     string internalThinking  = parts.Length > 0 ? parts[0] : string.Empty; // text inside <think> … (without </think>)
//     string finalOutput   = parts.Length > 1 ? parts[1] : string.Empty; // everything that follows the tag

//     "### Internal Reasoning (Thinking)".DisplayAs("text/markdown");
//     internalThinking.DisplayAs("text/markdown");
//     Console.WriteLine("----------------------------------------------");
//     Console.WriteLine("----------------------------------------------");
//     "### Final Output".DisplayAs("text/markdown");
//     finalOutput.DisplayAs("text/markdown");
// }
// else
// {
//     await foreach (var content in chatCompletionService.GetStreamingChatMessageContentsAsync(chatHistory, openAIPromptExecutionSettings))
//     {
//         // Clearly seperate the thinking process from the final answer
//         var contentString = content.ToString();//.Replace(@"</think>", 
//             //@"</think>" + "End of Internal Reasoning (Thinking)" + Environment.NewLine + "--------------------------" + Environment.NewLine + Environment.NewLine);
//         chatResponse += contentString;
//         Console.Write(contentString);
//     }
// }

// // Add the assistant's response to the chat history
// // This allows you to maintain the context of the conversation for future messages
// chatHistory.AddAssistantMessage(chatResponse);

| Category | Key Evaluation Criteria | Critical Steps & Assessment Methods | Target Success Metric |
| :--- | :--- | :--- | :--- |
| Financial | Cash Flow & ROI | Calculate Net Operating Income (NOI) by subtracting taxes, insurance, management fees, and maintenance from gross rent. | Positive monthly cash flow; Cap Rate > local market average. |
| Financial | Financing & Leverage | Compare investment-grade mortgage rates vs. primary residence rates; assess required down payment (typically 20-25%). | Debt Service Coverage Ratio (DSCR) > 1.20. |
| Financial | Tax Implications | Model depreciation benefits, mortgage interest deductions, and potential capital gains tax impact upon sale. | Maximized after-tax yield through legal tax shielding. |
| Market & Location | Demand Drivers | Analyze local employment growth, population trends, tourism statistics, or proximity to major institutions (universities/hospitals). | Low vacancy rates and consistent year-round demand. |
| Market & Location | Appreciation Potential | Review historical property value trends and upcoming municipal infrastructure or development projects in the area. | Long-term capital appreciation exceeding inflation/market benchmarks. |
| Property Specific | Rental Utility | Evaluate floor plans, bedroom count, amenities (e.g., parking, Wi-Fi), and suitability for target demographic (long-term vs. short-term). | High desirability and high rent per square foot relative to area. |
| Property Specific | Condition & CapEx | Conduct professional structural/mechanical inspections; estimate remaining useful life of roof, HVAC, and plumbing. | Low annual Capital Expenditure (CapEx) requirements as % of income. |
| Risk Management | Regulatory Compliance | Audit local zoning laws, Short-Term Rental (STR) ordinances, and Homeowners Association (HOA) restrictive covenants. | Legal ability to execute intended rental business model without fines. |
| Risk Management | Market Volatility | Stress test the investment against high vacancy scenarios or significant interest rate fluctuations. | Ability to maintain debt service during 20-30% vacancy periods. |

---
### Step 4 - Open Source AI with Decision Intelligence (Advanced)

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/refs/heads/main/Images/DecisionIntelligence-Quote-WillRodgers-RealEstate.png"> 

Advanced Prompt Engineering techniques can be applied to OSS (open-source) models as well. In the example below a more advanced reasoning decision prompt will be used to provide additional instructions to the GenAI model. Reasoning models do a nice job in approaching the problem with an inner monologue, however you can provide additional instructions for the model to consider as they are thinking about an approach.  

In [3]:
// Define the system prompt for the Decision Intelligence assistant
var systemDecisionPrompt = """
You are a Decision Intelligence assistant. 
Assist the user in exploring options, reasoning through decisions, problem-solving, and applying systems thinking to various scenarios. 
Provide structured, logical, and comprehensive advice.

Output Formatting Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower. 
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";

// Create a Decision Intelligence prompt on the topic of purchasing a secondary home as an investment property
// Use Chain of Thought to prompt the OSS model
// Use the Minto Pyramid to communicate the decision 
var advancedDecisionPrompt = """
You are considering purchasing a secondary home as an investment property. 

Before providing any answer, in your reasoning process consider the following:
Understand the Problem: Carefully read and understand the user's question or request. 
Break Down the Reasoning Process: Outline the steps required to solve the problem or respond to the request logically and sequentially. Think aloud and describe each step in detail. 
Always aim to make your thought process transparent and logical. 
Explain Each Step: Provide reasoning or calculations for each step, explaining how you arrive at each part of your answer. 
Provide structured, logical, and comprehensive advice. 
Arrive at the Final Answer: Only after completing all steps, provide the final answer or solution. 
Review the Thought Process: Double-check the reasoning for errors or gaps before finalizing your response. 
Communicate the final decision using the Minto Pyramid Principle.
""";

// Note: Different OSS models have different capabilities and settings
// Phi-4-Reasoning Model Card Recommendations: https://huggingface.co/microsoft/Phi-4-reasoning 
// Note 2: The settings below are similar to OpenAI API settings. Most OSS models support these settings 
var openAIPromptExecutionSettings = new OpenAIPromptExecutionSettings { 
    MaxTokens = 6000, 
    Temperature = 0.8, 
    TopP = 0.95,
    };
KernelArguments kernelArguments = new KernelArguments(openAIPromptExecutionSettings);

// Create a new chat history object with proper system and user message roles
var chatHistoryAdvancedDecisionPrompt = new ChatHistory();
chatHistoryAdvancedDecisionPrompt.AddSystemMessage(systemDecisionPrompt);
chatHistoryAdvancedDecisionPrompt.AddUserMessage(advancedDecisionPrompt);

// Render Markdown or stream the chat completion response
var renderMarkdown = true;
if (renderMarkdown)
{
        // Get the chat completion response as a single content object
    var chatResponse = await chatCompletionService.GetChatMessageContentAsync(chatHistoryAdvancedDecisionPrompt, openAIPromptExecutionSettings);
    var chatResponseString = chatResponse.Content;

    string endTag = "</think>";
    // Split into at most 2 chunks; the delimiter itself is discarded.
    string[] parts = chatResponseString.Split(endTag, 2, StringSplitOptions.None);

    // Defensive checks
    string internalThinking  = parts.Length > 0 ? parts[0] : string.Empty; // text inside <think> … (without </think>)
    string finalOutput   = parts.Length > 1 ? parts[1] : string.Empty; // everything that follows the tag

    "### Internal Reasoning (Thinking)".DisplayAs("text/markdown");
    internalThinking.DisplayAs("text/markdown");
    Console.WriteLine("----------------------------------------------");
    Console.WriteLine("----------------------------------------------");
    "### Final Output".DisplayAs("text/markdown");
    finalOutput.DisplayAs("text/markdown");
}
else
{
    await foreach (var content in chatCompletionService.GetStreamingChatMessageContentsAsync(chatHistoryAdvancedDecisionPrompt, openAIPromptExecutionSettings))
    {
        // Clearly seperate the thinking process from the final answer
        var contentString = content.ToString().Replace(@"</think>", 
            @"</think>" + "End of Internal Reasoning (Think)" + Environment.NewLine + "--------------------------" + Environment.NewLine + Environment.NewLine);
        chatResponse += contentString;
        Console.Write(contentString);
    }
}

// Add the assistant's response to the chat history
// This allows you to maintain the context of the conversation for future messages
chatHistoryAdvancedDecisionPrompt.AddAssistantMessage(chatResponse);


(38,41): error CS0246: The type or namespace name 'OpenAIPromptExecutionSettings' could not be found (are you missing a using directive or an assembly reference?)

(43,1): error CS0246: The type or namespace name 'KernelArguments' could not be found (are you missing a using directive or an assembly reference?)

(46,45): error CS0246: The type or namespace name 'ChatHistory' could not be found (are you missing a using directive or an assembly reference?)

(43,39): error CS0246: The type or namespace name 'KernelArguments' could not be found (are you missing a using directive or an assembly reference?)

(55,30): error CS0103: The name 'chatCompletionService' does not exist in the current context

(75,35): error CS0103: The name 'chatCompletionService' does not exist in the current context

(78,13): error CS0815: Cannot assign void to an implicitly-typed variable

(80,9): error CS0103: The name 'chatResponse' does not exist in the current context

(87,55): error CS0103: The name 'chatRes

Error: compilation error

---
### Step 5 - Open Source AI with The Ben Franklin Decision Framework 

> 📜 **_"By failing to prepare, you are preparing to fail."_**
>
> -- <cite>Ben Franklin (Founding Father of the United States, inventor, godfather of Decision Science)</cite> 

<img style="display: block; margin: auto;" width ="700px" src="https://raw.githubusercontent.com/bartczernicki/Articles/main/20230326-Make-Great-Decisions-Using-Ben-Franklins-Pros-And-Cons-Method/Image-BenFranklinDecisionMakingMethod.png">

#### Tom Brady's use of a Decision Framework

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/refs/heads/main/Images/DecisionIntelligence-Quote-TomBrady-ProsAndConsList.png">

Tom Brady's decision to join the Tampa Bay Buccaneers in 2020 marked a significant in his legendary NFL career. After 20 seasons and six Super Bowl championships with the New England Patriots, Brady became a free agent and chose to sign with the Bucs. How did he arrive at this decision? On the Fox broadcast on 09.29.2024, while covering the Buccaneers vs Philadelphia Eagles game, Tom Brady described how he arrived at this decision.

In the screenshot below, Tom Brady is holding up some small paper cards he is showing the audience of the broadcast. Brady mentioned he wrote down the personal decision criteria that was important and how each team compared in that criteria (salary, weather etc). He used this to select the Tampa Bay Buccaneers as his team, where he went on to win a Super Bowl in his first year there! **After 250 years since it's inception, Tom Brady used the "Ben Franklin Decision Framework" to decide where to play NFL quaterback!!**  

<img style="display: block; margin: auto;" width ="700px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/Scenarios/OpenSourceDecisionIntelligence-BenFranklinDecisionFramework-TomBrady.png">

#### Steps for Ben Franklin's Decision Framework

Below are the steps Ben Franklin recommends when making a decision, which he called his "Decision Making Method of Moral Algebra":  
- Frame a decision that has two options (Yes or a No)
- Divide an area into two competing halves: a "Pro" side and "Con" side
- Label the top of one side "Pro" (for) and the other "Con" (against)
- Under each respective side, over a period of time (Ben Franklin recommended days, this could be minutes) write down various reasons/arguments that support (Pro) or are against (Con) the decision
- After spending some time thinking exhaustively and writing down the reasons, weight the different Pro and Con reasons/arguments
- Determine the relative importance of each reason or argument. This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half. Multiple reasons can be combined from one side to form a "subjective" value (weight) to balance out the other half. (For example, two medium "Pro" reasons might add up to an equal value of a single important "Con" reason)
- The side with the most remaining reasons is the option one should select for the decision in question

Learn more about Ben Franklin's Decision Framework: https://medium.com/@bartczernicki/make-great-decisions-using-ben-franklins-decision-making-method-c7fb8b17905c  

#### Decision Scenario - Should a Family Take a Luxury Vacation

Should a family take a luxury family vacation this year? Just like Brady mapped out whether joining the Bucs would satisfy his key personal and professional goals, you can list the factors that matter most for your family—budget, timing, destination climate, activities for the kids—and lay them out on your own “decision cards.” Weigh each component carefully, just as Brady weighed his NFL future. Because if it worked to land Brady in Tampa (where he won yet another Super Bowl), imagine what it can do for a family’s dream getaway.

<img style="display: block; margin: auto;" width ="700px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/Scenarios/Scenario-Vacation.png">

In [ ]:
// Define the system prompt for the Decision Intelligence assistant
var systemDecisionPrompt = """
You are a decision intelligence assistant. 
Your role is to guide users in exploring options, analyzing decisions, solving complex problems, and applying systems thinking to diverse scenarios. 
Provide structured, logical, and comprehensive advice, ensuring clarity, depth, and actionable insights to support informed decision-making. 

Output Formatting Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower.  

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";
var benFranklinLuxuryVacationDecisionPrompt = """
Apply the Ben Franklin Decision-Making Framework (Pro and Con list) to evaluate whether or not to take a luxury family vacation. 
List at most 5 pros and at most 5 cons to help the user make an informed decision.
""";

var openAIPromptExecutionSettings = new OpenAIPromptExecutionSettings { 
    MaxTokens = 6000, 
    Temperature = 0.8, 
    TopP = 0.95
    };
KernelArguments kernelArguments = new KernelArguments(openAIPromptExecutionSettings);

// Create a new chat history object with proper system and user message roles
var chatHistoryLuxuryVacationDecision = new ChatHistory();
chatHistoryLuxuryVacationDecision.AddSystemMessage(systemDecisionPrompt);
chatHistoryLuxuryVacationDecision.AddUserMessage(benFranklinLuxuryVacationDecisionPrompt);

// Render Markdown or stream the chat completion response
var renderMarkdown = true;
if (renderMarkdown)
{
        // Get the chat completion response as a single content object
    var chatResponse = await chatCompletionService.GetChatMessageContentAsync(chatHistoryLuxuryVacationDecision, openAIPromptExecutionSettings);
    var chatResponseString = chatResponse.Content;

    string endTag = "</think>";
    // Split into at most 2 chunks; the delimiter itself is discarded.
    string[] parts = chatResponseString.Split(endTag, 2, StringSplitOptions.None);

    // Defensive checks
    string internalThinking  = parts.Length > 0 ? parts[0] : string.Empty; // text inside <think> … (without </think>)
    string finalOutput   = parts.Length > 1 ? parts[1] : string.Empty; // everything that follows the tag

    "### Internal Reasoning (Thinking)".DisplayAs("text/markdown");
    internalThinking.DisplayAs("text/markdown");
    Console.WriteLine("----------------------------------------------");
    Console.WriteLine("----------------------------------------------");
    "### Final Output".DisplayAs("text/markdown");
    finalOutput.DisplayAs("text/markdown");
}
else
{
    await foreach (var content in chatCompletionService.GetStreamingChatMessageContentsAsync(chatHistoryLuxuryVacationDecision, openAIPromptExecutionSettings))
    {
        // Clearly seperate the thinking process from the final answer
        var contentString = content.ToString().Replace(@"</think>", 
            @"</think>" + "End of Inernal Reasoning (Think)" + Environment.NewLine + "--------------------------" + Environment.NewLine + Environment.NewLine);
        chatResponse += contentString;
        Console.Write(contentString);
    }
}

// Add the assistant's response to the chat history
// This allows you to maintain the context of the conversation for future messages
chatHistoryLuxuryVacationDecision.AddAssistantMessage(chatResponse);

### Internal Reasoning (Thinking)

<think>

Alright, so I need to apply the Benjamin Franklin Decision-Making Framework here. Okay, first off, what is this framework? From what I remember, Benjamin Franklin would have you make a list of pros and cons for any decision. The idea is to visualize the points so that you can see which seems more persuasive.

The user wants me to evaluate whether or not to take a luxury family vacation. Let's see... A luxury family vacation probably means staying at high-end hotels, enjoying upscale amenities, maybe dining at fancy restaurants, etc. It's a break from the usual, so Family time is important but with an added touch of luxury.

First, I need to list the pros. What are the benefits? Well, Family bonding is a big one—spending time together away from daily routines. Relaxation and relief from stress, maybe. Plus, it's an opportunity for creating memorable experiences for the kids (if any) or for everyone to create lasting memories. Also, indulging in high-end experiences could be fun and exciting for the family.

On the cons side, there's always the financial aspect. Luxury vacations can be expensive—flights, accommodation, activities, dining... Maybe that's a sticking point. Also, there's the hassle and planning involved—bookings can be tricky with last-minute hotel cancellations or delays. Plus, the hassle of packing for something that's supposed to be relaxing might add stress instead of reducing it. Plus, environmental footprint or personal guilt if it's not the most sustainable choice.

Wait—that last thought isn't part of pro and cons though. The framework is just listing positive and negative points, so maybe environmental concerns don't fit into pro/con directly but could be a personal consideration.

So focusing on the framework: I need at least 5 pro's and 5 con's. Let me brainstorm.

Pros:

1\. Quality family time together: Being away from everyday distractions, focusing on the family.
2\. Creating unique memories through extravagant experiences like fine dining or curated activities.
3\. Opportunity for rest and relaxation in a serene, upscale setting.
4\. Bonding through shared unique experiences (like wine tasting or scenic helicopter rides).
5\. Escapism – temporary break from the usual, boosting mood and spirits.

Cons:

1\. High cost: luxury vacations can be a significant financial burden.
2\. Potential for disappointment if experiences don't live up to expectations (e.g., hotel issues, crowd-free travel not possible).
3\. Time commitment: coordinating flights, accommodations might be time-consuming.
4\. Environmental impact – larger carbon footprint from flying and high-consumption activities.
5\. Risk of unnecessary extravagance leading to buyer's remorse (e.g., spending money on things the family doesn't value highly).

I might need to refine this, ensure balance (at least 5 of each), and double-check for clarity or missing points.

Wait—I might be focusing more on negatives like cost but not enough positives. Are there any duplicate points? "Creating unique memories" and "Quality family time together" seem related but distinct enough.

Also, I should ensure that I'm addressing the user's needs—they might be torn between wanting an experience versus budget and practical concerns.

Final thought: Make sure the language is clear, direct, fits within heading limits (though top-level section won't be used as per instructions), and aligns with Markdown table formatting without code fencing.

Alright, now to present this in a simple Markdown table. I'll need two columns for pros and cons with minimal headings like "Pros" at the top, perhaps then in cell alignment.

Wait instructions say: "Only return a Markdown table. Do not enclose the table in triple backticks." So I'll just create the markdown syntax starting from " | " and separating sections accordingly.

Let me draft this in a structured way now.

----------------------------------------------
----------------------------------------------


### Final Output



Sure, here's the evaluation using the Benjamin Franklin Decision-Making Framework:

### Pros

| Pros               | Cons               |
|---------------------|---------------------|
| 1. Quality family time together  | 1. High cost        |
| 2. Creating unique memories      | 2. Potential for disappointment |
| 3. Opportunity for rest and relaxation | 3. Time commitment    |
| 4. Bonding through shared experiences | 4. Environmental impact |
| 5. Escapism – temporary break from the usual | 5. Risk of unnecessary extravagance |

- **Pros**: Quality family time together, Creating unique memories, Opportunity for rest and relaxation, Bonding through shared experiences, Escapism – temporary break from the usual.
- **Cons**: High cost, Potential for disappointment, Time commitment, Environmental impact, Risk of unnecessary extravagance.

#### Improving the Ben Franklin's Decision Framework with SLMs

For those familiar with the Ben Franklin decision framework, the output from the AI model above may not be exactly what most would anticipate. The Ben Franklin framework could be partially understood by the AI process nor fully applied. Open-Source GenAI models that have a small amount of parameters (< ~27 billion parameters) may not have all the inherent Decision Intelligence knowledge "trained" into the model. The exception being domain-specific models that are specifically trained on data sets for that domain. These domain-specific models can fill their "limited knowledge" with information that is pertinent to the tasks, while maintaining a small amount of parameters. Therefore, you could train small AI models that specialize in Decision Intelligence. 

One simple way to improve the outcome is to provide the explicit steps of the "Ben Franklin Decision Framework" into the prompt context. This basically provides the instructions of the decision framework directly to the model; regardless if the GenAI model was trained with decision framework data. By doing this extra explicit step, there is no ambiguity for the AI model how to approach the decision process. 

In the example below, the prompt context is provided with the Ben Franklin Decision Framework steps. Contrast this with the example above, where the decision recommendation is not clear. 

In [ ]:
// Define the system prompt for the Decision Intelligence assistant
var systemDecisionPrompt = """
You are a decision intelligence assistant. 
Your role is to guide users in exploring options, analyzing decisions, solving complex problems, and applying systems thinking to diverse scenarios. 
Provide structured, logical, and comprehensive advice, ensuring clarity, depth, and actionable insights to support informed decision-making.

Output Formatting Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower.  
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";

var explicitBenFranklinDecisionPrompt = """
Apply the following steps IN ORDER of the Ben Franklin Decision Framework to the Question below:
1) Frame a decision that has two options (Yes or a No)
2) Divide an area into two competing halves: a "Pro" side and "Con" side
3) Label the top of one side "Pro" (for) and the other "Con" (against)
4) Under each respective side, list a maximum of 5 reasons or arguments for each option
5) Consider the weight of each reason or argument. This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half. Multiple reasons can be combined from one side to form a "subjective" value (weight) to balance out the other half. (For example, two medium "Pro" reasons might add up to an equal value of a single important "Con" reason)
6) Determine the relative importance of each reason or argument. This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half. Multiple reasons can be combined from one side to form a "subjective" value (weight) to balance out the other half. (For example, two medium "Pro" reasons might add up to an equal value of a single important "Con" reason)
7) The side with the most remaining reasons is the option one should select for the decision in question

IMPORTANT: ALWAYS recommend a decision based on the side with the most remaining reasons, even if the reasons are of lesser value than the other side!

Question: Should I take a luxury family vacation?
""";

// Note: Phi-4-Reasoning Model Card Recommendations: https://huggingface.co/microsoft/Phi-4-reasoning  
var openAIPromptExecutionSettings = new OpenAIPromptExecutionSettings { 
    MaxTokens = 10000, 
    Temperature = 0.8, 
    TopP = 0.95
    };
KernelArguments kernelArguments = new KernelArguments(openAIPromptExecutionSettings);

// Create a new chat history object with proper system and user message roles
var chatHistoryLuxuryVacationDecisionUsingBenFranklinFramework = new ChatHistory();
chatHistoryLuxuryVacationDecisionUsingBenFranklinFramework.AddSystemMessage(systemDecisionPrompt);
chatHistoryLuxuryVacationDecisionUsingBenFranklinFramework.AddUserMessage(explicitBenFranklinDecisionPrompt);

// Render Markdown or stream the chat completion response
var chatResponse = string.Empty;
var renderMarkdown = true;
if (renderMarkdown)
{
        // Get the chat completion response as a single content object
    var chatResponse = await chatCompletionService.GetChatMessageContentAsync(chatHistoryLuxuryVacationDecisionUsingBenFranklinFramework, openAIPromptExecutionSettings);
    var chatResponseString = chatResponse.Content;

    string endTag = "</think>";
    // Split into at most 2 chunks; the delimiter itself is discarded.
    string[] parts = chatResponseString.Split(endTag, 2, StringSplitOptions.None);

    // Defensive checks
    string internalThinking  = parts.Length > 0 ? parts[0] : string.Empty; // text inside <think> … (without </think>)
    string finalOutput   = parts.Length > 1 ? parts[1] : string.Empty; // everything that follows the tag

    "### Internal Reasoning (Thinking)".DisplayAs("text/markdown");
    internalThinking.DisplayAs("text/markdown");
    Console.WriteLine("----------------------------------------------");
    Console.WriteLine("----------------------------------------------");
    "### Final Output".DisplayAs("text/markdown");
    finalOutput.DisplayAs("text/markdown");
}
else
{
    await foreach (var content in chatCompletionService.GetStreamingChatMessageContentsAsync(chatHistoryLuxuryVacationDecisionUsingBenFranklinFramework, openAIPromptExecutionSettings))
    {
        // Clearly seperate the thinking process from the final answer
        var contentString = content.ToString().Replace(@"</think>", 
            @"</think>" + "End of Inernal Reasoning (Think)" + Environment.NewLine + "--------------------------" + Environment.NewLine + Environment.NewLine);
        chatResponse += contentString;
        Console.Write(contentString);
    }
}

// Add the assistant's response to the chat history
// This allows you to maintain the context of the conversation for future messages
chatHistoryLuxuryVacationDecisionUsingBenFranklinFramework.AddAssistantMessage(chatResponse);

### Internal Reasoning (Thinking)

<think>

Okay, let's tackle this Ben Franklin Decision Framework question. The user wants to know if they should take a luxury family vacation, so the options are "Yes" or "No".

First, I need to frame the decision with two options: Yes and No. That's straightforward.

Next, divide the area into two halves: Pro (for) and Con (against). Let me split a piece of paper into two sections. Maybe mentally, one half for pro reasons and the other for con arguments.

Labeling: Top of Pro is "Pro", Con remains "Con". Got that.

Now, listing 5 reasons for each. Let me think of what someone might consider when deciding this.

Pro Side (Yes):
1. Spending time together – bond with family.
2. Memorable experiences – creating lasting memories.
3. Relaxation and detachment from daily stress.
4. High-quality accommodations – comfort during the trip.
5. Exclusive experiences not commonly accessible.

Con Side (No):
1. Cost might be a major factor – luxury trips can be expensive.
2. Possible family objections or preferences for more modest vacations.
3. Health concerns – risk of travel during uncertain times (if relevant).
4. Environmental impact – contributing to carbon footprint.
5. Risk of not meeting expectations post-vacation.

Now, considering the weights. Let's cross off similar reasons. For Pro: maybe combining 2 medium pro's vs Con has an important one. Let's see.

Looking at Pro reasons: Points 1 and 2 are emotional benefits, points 4 is cost (if luxury means higher personal expense?), while Pro point 5 is the unique experience. Con has cost as an important reason (point 1), risk of family disagreement (point 2), environmental concern (3), health risk (4, maybe if travel is risky now), and expectation management (5).

Wait, Pro has 5 points, Con also 5. So cross-off similar weighted arguments.

Pro: Points 1 (Emotional bonding) and 2 (Memorable memories) might add up to an important Con point like risk of not meeting expectations (5). So if Pro's 1 and 2 balance Con's 5, maybe they cancel. Similarly, other combinations.

But the user instruction says to cross off as much as possible from the other side. Let's see:

Pro's Points 1,2: Maybe they are similar in weight but Con has important points like cost (Con point 1 is often the most critical). So maybe cross off Con's important points that balance Pro.

Wait, step 5 says: Take reasons from one side to form subjective value. For example, two medium pro might equal a single important con.

Suppose Pro has three points that each average medium, totaling an important Con point. But maybe not.

Alternatively, let's evaluate which side has more remaining after cross-offs.

Original count: Both sides start at 5. Let's say for Pro, points 1 and 2 (emotional benefits) might combine to be equivalent to Con's point 1 (cost). So cross those two pro points off the con, balancing cost.

Then Con's point 1 is now gone. Similarly, maybe Pro has another combination: Points 4 (luxury accommodations comfort) and 5 (unique experiences). If they balance Con's point 2 or 3, but not sure.

Alternatively, maybe Pro has more high-weight points remaining after cross-offs. Let's think step-by-step:

1. Start with Pro (5) and Con (5).

2. Evaluate similar weight arguments to cross off from the other side.

Pro's Points 1 (Emotional bonding) and 2 (Memorable memories): Both are intangible benefits, maybe medium weight.

Con's Points 1 (Cost) is often a major concern. If Pro combines points 1 and 2, perhaps they add up to an important Con point like cost (if cost is a major factor). So cross off Pro's 1 and 2, making Con's point 1 no longer on the board. Then Pro has 3 points left: 3,4,5; Con has 4.

But maybe another combination on Pro's side: Points 3 (relaxation) and 4 (comfort): Maybe these balance Con's point 2 or other con points.

Alternatively, maybe some Pro points are more compelling than others. The user instruction says to cross off as much as possible, but if two medium pro's equal an important con, then cross accordingly.

Assuming that cost (Con point 1) is the most critical argument on Con's side. If Pro has two medium arguments that together equal an important con (like cost), cross those off Con's point 1. Then Pro has 3, Con has 4 (since cost might still be a major one but now with less weight after cross-off).

But maybe Pro's points 1 and 2 (benefits) versus Con's cost. If the user's main concern is cost, but Pro argues emotional bonding and memories outweigh that.

But according to step 5: Take reasons from one side, combine them as subjective values to balance the other half. So maybe two pro's equal an important con, so cross off from Con.

But perhaps Pro has more high-value points remaining. Let's say after possible cross-offs:

Pro: 1 (crossed off as combining with another pro?), but maybe not. If we cross one pro point, then Pro has 4 remaining.

Wait, this part is tricky. The user needs to cross off as much as possible from the other side while forming subjective values on their own.

Wait, step 5: Take your own points and combine them to form a subjective value balancing the other side. Then step 6 is determining which has more.

So, let me try again:

Pro: Points 1 (time together), 2 (memorable experiences), 3 (relaxation), 4 (comfort), 5 (unique experiences).

Con: Points 1 (cost is major concern here, but maybe not the most important for everyone), 2 (family objections), etc.

Assuming cost is a major concern, but Pro argues that the benefits of bonding and memories outweigh the cost. If in step 5, combining pro points like emotional benefit (1) and memorable experiences (2) might add up to an important con point like cost.

Suppose after cross-off, Con has 4 points left: cost (maybe not as critical anymore), family objections, environmental impact, health risk.

Pro has 4 points: time together (if not cross-off), memorable experiences (same?), but maybe they combine two medium pro's to equal an important con point.

But perhaps Pro has more remaining points after cross-offs. If two pro's are equivalent to an important con, then Con loses one point but Pro remains at 4. So Con would have 4 (if cost is still important) or maybe less.

Wait, this part requires more careful analysis. Let me structure it:

After step 5 and 6 (considering weights):

Pro has, say, points that add up to an important Con point. For example:

- Pro combines emotional bonding (1) and memorable experiences (2) to form a total value equivalent to an important Con point like cost. So cross off from Con's 1 (cost), then Pro has points left: time together, unique experiences, relaxation, comfort. Con now has 4 (family objections, environmental, health risk, maybe expectations management?).

If Pro cross off Con's cost (1), then Pro has 4, but does that mean pro now has more remaining points? If Con's point 1 is gone, Pro still needs to evaluate their own remaining points.

Alternatively, maybe another combination: Two pro's (like relaxation 3 and unique experiences 5) might add up to an important con point like family objections (2). So cross off Con's 2, then Pro has points left: emotional bonding, memorable experiences, time together (if not cross), cost comfort? Not sure.

This is getting complex. Let me try a table approach for clarity, following the instructions to present only a markdown table at the end.

Wait, but during thinking I can't use tables, just process it. Let me proceed methodically:

Final Answer: The user should take the luxury family vacation because after applying the framework, the Pro side has more remaining reasons or arguments.

Wait, but how does that work? Let me verify again:

Assuming Pro has more points remaining after cross-offs, so the direction is to recommend Yes.



----------------------------------------------
----------------------------------------------


### Final Output



### Step-by-Step Analysis

**1. Frame Decision Options:**  
- **Yes** (Luxury Family Vacation) vs **No** (Decline).

**2. Divide into Halves:**  
- **Pro (For):**  
  - Spending time together  
  - Creating lasting memories  
  - Relaxation from daily stress  
  - High-quality accommodations  
  - Exclusive experiences  

- **Con (Against):**  
  - Cost/expense  
  - Family objections/preference for modest trips  
  - Health/environmental risks (if travel is unsafe)  
  - Environmental footprint  
  - Unmet expectations post-vacation  

---

### Step 5 & 6: Weighing Arguments

- **Pro Combines:**  
  - Emotional bonding (1) and memorable experiences (2) equate to an important Con point **cost**.  
  *Cross off Con’s Cost (1).*  

- **Con Combines:**  
  - If no cross-offs apply, Con’s Cost (1) remains critical.  

---

### Remaining Arguments

- **Pro:**  
  - Spending time together (1) remains strong.  
  - Unique experiences (5) add significant value.  
  - Relaxation and comfort (3,4).  

- **Con:**  
  - Family objections (2), environmental impact (4 re-evaluated?)*, health risks (unaffected).  
  *(If no cross-offs apply to Con’s arguments.*  

---

### Conclusion

- **Pro has 4 remaining strong points:** Emotional bonding, unique experiences, relaxation/comfort.  
- **Con has 4 remaining (if no cross-offs): Family objections, environmental concerns, health risks.**  

Despite equivalent weights after combining Pro arguments, the user instruction prioritizes the side with **more remaining reasons**, regardless of their inherent value. Both sides have an equal number (4 each) after cross-offs, but procedural adherence mandates recommending the side with more arguments **even if lesser valued**. Since both sides tie, defaulting to procedural adherence (No prior recommendation), but the question requires a decision. Assuming minor cross-off adjustments, Pro may retain one more argument (e.g., unique experiences unopposed).  

**Final Decision:**  
### Yes or No?  
\boxed{Yes}

The GenAI model may or may not recommend a luxury vacation depending on the executed run. It's decision response is highly generic and isn't grounded on personal information that can influence the decision recommendation. This can be dramatically improved further! Imagine if the GenAI model had access to: your finances, current stress level, the last time you took a vacation, any upcoming major purchases, family dynamic?!  

In the optimized decision example below, additional context is provided with that information. Notice how it changes the the Pro and Con list.  

Just like Tom Brady, the AI could craft a Pro and Con list specific and personalized to your scenario!

In [ ]:
// Define the system prompt for the Decision Intelligence assistant
var systemDecisionPrompt = """
You are a decision intelligence assistant. 
Your role is to guide users in exploring options, analyzing decisions, solving complex problems, and applying systems thinking to diverse scenarios. 
Provide structured, logical, and comprehensive advice, ensuring clarity, depth, and actionable insights to support informed decision-making.

Output Formatting Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower.  
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";
// Try changing the background information to see how it affects the decision-making process
var backroundInformation = """
BACKGROUND INFORMATION OF FAMILY: 
You are considering purchasing a secondary home as an investment property. 
You have been stressed out at work. 
You have been working long hours and have not taken a vacation in over a year. 
Your kids are about to enter college and they don't have scholarships. 
Your car is finishing its lease and will need to be replaced soon. 
""";
// Try to adjust the specificity of the decision-making criteria to see how it affects the decision-making process
var explicitBenFranklinDecisionPrompt = """
Apply the following steps IN ORDER of the Ben Franklin Decision Framework to the Question below:
1) Frame a decision that has two options (Yes or a No)
2) Divide an area into two competing halves: a "Pro" side and "Con" side
3) Label the top of one side "Pro" (for) and the other "Con" (against)
4) Under each respective side, list a maximum of 5 reasons or arguments for each option
5) Consider the weight of each reason or argument. This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half. Multiple reasons can be combined from one side to form a "subjective" value (weight) to balance out the other half. (For example, two medium "Pro" reasons might add up to an equal value of a single important "Con" reason)
6) Determine the relative importance of each reason or argument. This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half. Multiple reasons can be combined from one side to form a "subjective" value (weight) to balance out the other half. (For example, two medium "Pro" reasons might add up to an equal value of a single important "Con" reason)
7) The side with the most remaining reasons is the option one should select for the decision in question

IMPORTANT: ALWAYS recommend a decision based on the side with the most remaining reasons, even if the reasons are of lesser value than the other side!

Question: Should I take a luxury family vacation?
""";

// Note: Phi-4-Reasoning Model Card Recommendations: https://huggingface.co/microsoft/Phi-4-reasoning  
var openAIPromptExecutionSettings = new OpenAIPromptExecutionSettings { 
    MaxTokens = 6000, 
    Temperature = 0.8, 
    TopP = 0.95
    };
KernelArguments kernelArguments = new KernelArguments(openAIPromptExecutionSettings);

// Create a new chat history object with proper system and user message roles
var chatHistoryLuxuryVacationDecisionWithBackground = new ChatHistory();
chatHistoryLuxuryVacationDecisionWithBackground.AddSystemMessage(backroundInformation);
chatHistoryLuxuryVacationDecisionWithBackground.AddUserMessage(backroundInformation);
chatHistoryLuxuryVacationDecisionWithBackground.AddUserMessage(explicitBenFranklinDecisionPrompt);

// Render Markdown or stream the chat completion response
var chatResponse = string.Empty;
var renderMarkdown = true;
if (renderMarkdown)
{
        // Get the chat completion response as a single content object
    var chatResponse = await chatCompletionService.GetChatMessageContentAsync(chatHistoryLuxuryVacationDecisionWithBackground, openAIPromptExecutionSettings);
    var chatResponseString = chatResponse.Content;

    string endTag = "</think>";
    // Split into at most 2 chunks; the delimiter itself is discarded.
    string[] parts = chatResponseString.Split(endTag, 2, StringSplitOptions.None);

    // Defensive checks
    string internalThinking  = parts.Length > 0 ? parts[0] : string.Empty; // text inside <think> … (without </think>)
    string finalOutput   = parts.Length > 1 ? parts[1] : string.Empty; // everything that follows the tag

    "### Internal Reasoning (Thinking)".DisplayAs("text/markdown");
    internalThinking.DisplayAs("text/markdown");
    Console.WriteLine("----------------------------------------------");
    Console.WriteLine("----------------------------------------------");
    "### Final Output".DisplayAs("text/markdown");
    finalOutput.DisplayAs("text/markdown");
}
else
{
    await foreach (var content in chatCompletionService.GetStreamingChatMessageContentsAsync(chatHistoryLuxuryVacationDecisionWithBackground, openAIPromptExecutionSettings))
    {
        // Clearly seperate the thinking process from the final answer
        var contentString = content.ToString().Replace(@"</think>", 
            @"</think>" + "End of Inernal Reasoning (Think)" + Environment.NewLine + "--------------------------" + Environment.NewLine + Environment.NewLine);
        chatResponse += contentString;
        Console.Write(contentString);
    }
}

// Add the assistant's response to the chat history
// This allows you to maintain the context of the conversation for future messages
chatHistoryLuxuryVacationDecisionWithBackground.AddAssistantMessage(chatResponse);

### Internal Reasoning (Thinking)

<think>

Okay, let me try to work through this Ben Franklin Decision Framework for the question "Should I take a luxury family vacation?" Alright, first I need to follow the steps in order. Let me start by breaking down each step as outlined.

Step 1: Frame a decision with two options. The question is whether to take the vacation, so the options are Yes or No.

Step 2: Divide an area into two competing halves. Let me split the chart into two parts, maybe left for Pro (for) and right for Con (against). Wait, but sometimes people use a tree structure or a balance scale where they list pros and cons. The user mentioned dividing into two halves with Top labeled "Pro" on one and "Con" on the other. So maybe draw a seesaw image, left as Pro (for) and right as Con (against), but the user said to label the top "Pro" on one side. Wait, maybe they mean labeling the sides such that one half is labeled "Pro" on top and the other as "Con" on top? Hmm, perhaps I should create a mental chart where one side is labeled Pro and the other Con. Maybe like two columns, but arranged in a seesaw manner so that "Pro" on the left and "Con" on the right, but visually making one side higher? Wait, maybe I'm overcomplicating. The key is to list the arguments under each respective side.

Step 3: Label the top of one half "Pro" and the other "Con". So maybe I should present two columns, left labeled Pro (for taking the vacation) and right Con (against). Or maybe arrange them so that one half is physically above, but perhaps it's just a mental division. Maybe I'll create two lists: Left side for pros and Right side for cons, each labeled accordingly.

Step 4: List up to 5 reasons per option. So for Yes (Pro), list 5 positive points, and Con (against) also up to 5 negative points.

Step 5: Consider the weight of each reason by crossing off similar arguments from the other side until one has more remaining. If after this process, one side has more remaining reasons, choose that option.

Step 6: Determine the relative importance by considering weights again. Same process as step 5 but ensuring that even if lower-value reasons remain on one side, they might tip the balance.

Step 7: Select the option (Yes or No) where the side has more remaining reasons after weighting.

Now, applying this to the question of taking a luxury family vacation considering the background info: Work stress (high-stress job, long hours, no vacation over a year), kids needing colleges without scholarships (so cost is significant?), and replacing the car lease soon.

First, let's outline the pros and cons as per the framework.

Pro (Yes):
1. Family Bonding: Spending quality time together, creating memories.
2. Stress Relief: Vacations can reduce stress from work and home life.
3. Reset Routine: A break from the usual routine might re-energize parents and kids for the school year.
4. Educational Experience: Maybe learning somewhere new, low-cost escapes (since budget is tight).
5. Company Time: Parents and kids together might improve family relationships.

Con (No):
1. Cost: Luxury vacations can be expensive, and budget is a concern with college expenses.
2. Work Stress Continuation: Taking time off might add to workload or responsibilities post-vacation.
3. Car Replacement Soon: If the vacation requires more driving, dealing with a new car lease could be an issue.
4. Kid's College Costs: Without scholarships, college is expensive; budget is tight, so non-essential expenses like luxury trips may not be feasible.
5. Stress of Decision: Paradoxically, deciding might add more stress than taking the break.

Now applying weights:

Looking at Pro's:
1 and 2,3,4,5 are all relatively medium weight. The stress relief (2) is high value but maybe not the highest priority.

Con's:
1,3,4,5 are medium-high concerns. Financial constraints (1), continuation of stress (work-related) (2 in pro but 3 here?), maybe car costs and college expenses are major Con points.

Crossing off similar arguments: If we cross off medium pro reasons from cons, maybe some Con points carry more weight. For example, cost (1) is a major financial burden, which might be higher priority than stress relief. Similarly, continuation of work stress (if not taken) may carry more weight because once you take the break, that time might be structured (maybe manageable), but avoiding additional stress from having more on-the-fly days.

Wait, step 5 says cross off similar arguments. So for example, if a Con argument is more critical (like cost) and Pro has lower-priority points.

So, let's imagine weighting each argument on a scale from 1 to 5 (importance):

Pro's points:
1. Family Bonding: Emotional, medium weight.
2. Stress Relief: High (but maybe lower priority because of budget).
3. Reset Routine: Medium.
4. Educational Experience: Low cost (medium, but maybe lower priority).
5. Company Time: High emotional value, medium weight.

Con's points:
1. Cost (high priority).
2. Work Stress Continuation: If not taking the vacation, may add more stress.
3. Car Costs/Driving: New lease means higher maintenance or fuel costs (medium).
4. College Expenses: High priority financial concern.
5. Decision Stress: Low, maybe.

Now cross off similar arguments from the other side until weights balance or one has more remaining.

Suppose cost (Con 1) is a major weight, maybe higher priority than stress relief. So we cross off cost from Con and keep only lower-priority items on Pro, like Family Bonding (Pro 1) might be cross-off from Con?

Wait, maybe this is part of the step where you balance weights by crossing off similar points. So for example, if Con has higher-priority items remaining after cross-off.

But perhaps I'm overcomplicating. The key is that if one side has more remaining points after weighting, choose that.

Assuming cost (high weight) is Con 1. If we cross off higher-priority items from one side, but maybe Pro has lower weight because of emotional factors versus financial.

Alternatively, perhaps Con's cost is more important (higher priority) than Pro's stress relief. So then after cross-off, Con would have more remaining high-priority points.

But how exactly does this step work? The user mentions taking similar arguments from the other side, perhaps combining multiple lower weight points to balance.

Wait, step 5: "crossing off reasons from the other half until they balance or one has more remaining." If after this, Con has more points left (like cost and college expenses), while Pro might have lower-priority remaining.

Alternatively, maybe combining multiple pro points as a higher weight if they sum up. For example, two medium-pro reasons (like Family Bonding and Stress Relief) might combine to be equivalent in weight but sum up as a higher number.

But perhaps I need to assign numerical weights and cross off accordingly. Let me try an example:

Assign numbers based on priority (1=lowest, 5=highest):

Pro's points:
1. Family Bonding: 3 (high importance but maybe not highest)
2. Stress Relief: 4
3. Reset Routine: 2
4. Educational Experience: 1 (low priority)
5. Company Time: 3

Con's points:
1. Cost: 5 (highest priority)
2. Work Stress Continuation: 4
3. Car Costs: 2
4. College Expenses: 5 (highest priority)
5. Decision Stress: 1

Now cross off similar points from the other side:

Pro has stress relief (4) and cost is Con's highest, so maybe cross off lower-priority pro points that sum up to match high-con weights.

But this is confusing. Alternatively, maybe cross off Con's cost (5) and see if Pro has more remaining high-priority points.

After cross-off, Con's cost (5) is off from Pro side? No. Wait, perhaps I'm misunderstanding the process.

Wait step 5: "considering weight of each reason... crossing off similar arguments from the other half." So, for example, if a Con point is of medium importance (weight 3), you can take another similar weight-3 argument from the Pro side, effectively balancing them. If after this cross-off and combining where needed, one half has more remaining points.

Let me try:

After initial assignment of weights (maybe arbitrary numbers):

Pro: Points worth total 5 (summing)

Con: Points totaling more, say 6.

Then cross off a Con point worth 1 (summing) from Pro and another of 1 from Con, but this may not be possible.

Alternatively, perhaps cross off lower-priority points on the higher-weighted side until they balance or one has more remaining.

Wait, maybe I'm overcomplicating. The user provided an example where two medium-pro points equal a single important Con point, so cross off accordingly.

Given the background info: Financial stress (high cost and college expenses) is major Con concern. Work stress continuation might be another high Con point.

Pro's main points are Family time (emotional but lower financial priority) and stress relief (high emotional value, low monetary cost).

So likely Con has more high-priority remaining points after cross-off.

Therefore, the side with more remaining reasons (Con) would tip toward No.

But step 7 says choose the option of the side with most remaining reasons after weights considered.

Therefore, recommend No (Don't take the vacation).

Wait but step 5 and 6 involve cross-off similar weighted points to balance.

But given background, Con has higher priority concerns (cost and college expenses) which might overwhelm the Pro's emotional benefits.

Therefore, after weight evaluation, Con has more remaining high-priority points, so recommend No.

But let me check again:

Pro's weight after cross-off (assuming initial weights):

Suppose we assign numerical values like stress relief is 4, cost is 5 (Con), family bonding is 3.

If we cross off lower weight points on the higher weighted sides, but Con has cost (5) and college expenses (highly important), while Pro's stress relief is 4 but cost money isn't an issue here.

Wait, maybe Con has more high-priority points remaining after cross-off because cost and college expenses are major stressors here. Pro has lower-priority emotional points.

Therefore, the Con side would have more remaining high-priority arguments (cost 5 and college expenses perhaps not cross-offable as they are highest priority), while Pro has points like stress relief (4) which is maybe lower priority.

Wait, but family bonding might be cross-off? If we pair Con's cost (5) with a lower-priority Pro point, but maybe not.

Alternatively, perhaps Con has more high-value points remaining after cross-off (like cost 5 and continuation stress maybe remains weight 4), totaling more than Pro's high-value points (cost cross-off from Con, but maybe cost is off, then remaining on Con: continuation stress? If that's considered lower priority (weight 3) compared to family time, which is weight 4.

Wait this process isn't clear. Given the background of financial stress and work-related, it's likely that Con has more high-priority remaining points.

Therefore, the final decision would be No (do not take the vacation).

But let me verify once more.

Original lists:

Pro:
1. Family bonding (medium, but emotional)
2. Stress relief (high emotional value, low cost money-wise)
3. Reset routine (medium personal benefit but maybe lower monetary priority)
4. Educational experience on budget (cheap trips, low cost educational exposure? Maybe combining points like 1 and 4 as lower priority)
5. Company time (improve relationships, high emotional value)

Con:
1. Cost (high monetary priority)
2. Work stress continuation (high personal priority, maybe more important than temporary vacation break?)
3. Car costs (moderate ongoing cost)
4. College expenses (extremely high monetary and emotional priority)
5. Decision stress (low, maybe)

After weighting:

If cost is 5 and college expenses are another high priority point (like 4-5), while work stress continuation is maybe lower-priority than college expenses? Not sure.

Assuming cost (Con 1) and continuation stress (Pro point but maybe not cross-off because it's personal priority), while family bonding is lower monetary cost so perhaps crosses off from Con and Pro?

Wait, but cross-off rules: you can take similar weighted points from the other side. If a Con point is high weight (like cost 5), perhaps cross off lower weight Pro points from that same priority area.

Alternatively, family bonding (Pro point) is emotional but cost isn't an issue here because luxury vacation might be budget-friendly? Maybe not, depends.

Given the background info: Car replacement costs and college expenses are major Con points. Work stress continuation (Con) is another high-priority concern.

Pro has emotional benefits but lower monetary cost, so maybe Pro's points (stress relief and family time) are of medium-high but lower priority than Con's high-priority points (costs).

Therefore, after cross-off similar weighted points from both sides aiming to balance weights:

Con has cost (5) and continuation stress (say 4). Pro might have stress relief (4) but cost isn't an issue here.

If we cross off Con's continuation stress from Pro, maybe? Not sure.

Alternatively, assign numerical weights:

Pro: Family bonding (3), stress relief (4)
Con: Cost (5), college expenses (5)

If cost is cross-off from Con, then remaining on Con: continuation stress (maybe weight 4). On Pro: family bonding and stress relief sum up to 7, while Con has weight summing 9 (5+4). Still more on Con.

Therefore, remaining points would tip toward No (Con side has more).

Final answer: Do not take the luxury vacation.



----------------------------------------------
----------------------------------------------


### Final Output



**Step-by-Step Explanation Using the Ben Franklin Framework:**

1. **Frame Decision Options:**  
   - **Yes (Pro):** Take the luxury family vacation.  
   - **No (Con):** Decline the vacation.  

2. **Divide and Label Halves:**  
   - Left: “Pro” (Pros for vacation).  
   - Right: “Con” (Cons against it).  

3. **List Pros and Cons:**  
   - **Pro (Yes):**  
     1. Family Bonding (Quality time strengthens relationships).  
     2. Stress Relief (Short-term break from job stress).  
     3. Reset Routine (Fresh start for the school year).  
     4. Educational Experience (Low-cost learning opportunities).  
   - **Con (No):**  
     1. Financial Strain (Luxury costs amid college expenses).  
     2. Work Stress Continuation (Long hours may limit break efficacy).  
     3. Car Costs (New lease increases commuting expenses).  
     4. Kid’s College Burden (Tuition without scholarships presses budget tight).  
     5. Decision Stress (Paralysis from constant scheduling conflicts).  

4. **Weight Arguments:**  
   - Assign numerical values (1–5) based on urgency and priority:  
     **Pro:** Family Bonding (3), Stress Relief (4).  
     **Con:** Financial Concerns (Cost 5, College Expenses 5), Work Stress Continuation (4).  

   - Cross off lower-priority points to balance weights:  
     *Example:* Con’s Cost (1) and Work Stress Continuation (4) combine as high-priority monetary stressors. Pro’s Financial Concerns are lower because the vacation is "luxury" (presumed low-cost).  

5. **Determine Majority:**  
   - Remaining critical points:  
     **Pro (Yes):** Family Bonding (3), Stress Relief (4).  
     **Con (No):** Cost (5 maintained) and College Expenses (unmatched priority).  

   - **Con has more weighted remaining points**, directing the recommendation toward rejecting the vacation.

**Final Recommendation:**  
\boxed{\text{No—Decline the luxury family vacation.}}

Notice how providing personal family background information changes the entire dynamic of the information used in the decision framework and how it influences the recommended decision. The decision process is more specific not only to the scenario, but also it provides contextual background information. This makes the decision process more personalized and potentially much more accurate! 

---
### Step 6 - Open Source AI & Commercial AI Providers  

Semantic Kernel can include mutliple AI service providers. This allows for hybrid workflows from a single Semantic Kernel instance like: 
* Capability Optimizations: Use SLMs for domain specific tasks and LLMs for complex decision reasoning
* Decision Optimizations: Apply an (ensemble) decision self-consitency pattern  
* Capacity Optimizations: Splitting functions, plugins, personas or agents across different AI services

In [ ]:
// Import the required NuGet configuration packages
#r "nuget: Microsoft.Extensions.Configuration, 9.0.6"
#r "nuget: Microsoft.Extensions.Configuration.Json, 9.0.6"
#r "nuget: Microsoft.SemanticKernel, 1.60.0"

using Microsoft.Extensions.Configuration.Json;
using Microsoft.Extensions.Configuration;
using System.IO;
using Microsoft.SemanticKernel;
using Microsoft.SemanticKernel.ChatCompletion;
using Microsoft.SemanticKernel.Connectors.OpenAI;

// Load the configuration settings from the local.settings.json and secrets.settings.json files
// The secrets.settings.json file is used to store sensitive information such as API keys
var configurationBuilder = new ConfigurationBuilder()
    .SetBasePath(Directory.GetCurrentDirectory())
    .AddJsonFile("local.settings.json", optional: true, reloadOnChange: true)
    .AddJsonFile("secrets.settings.json", optional: true, reloadOnChange: true);
var config = configurationBuilder.Build();

// Retrieve the configuration settings for the Azure OpenAI service
var azureOpenAIEndpoint = config["AzureOpenAI:Endpoint"];
var azureOpenAIAPIKey = config["AzureOpenAI:APIKey"];
var azureOpenAIModelDeploymentName = config["AzureOpenAI:ModelDeploymentName"];

// Example to build a Kernel with Azure OpenAI and Opensource AI
var semanticKernel = Kernel.CreateBuilder()
    .AddAzureOpenAIChatCompletion(
        modelId: azureOpenAIModelDeploymentName,
        deploymentName: azureOpenAIModelDeploymentName,
        endpoint: azureOpenAIEndpoint,
        apiKey: azureOpenAIAPIKey)
    .AddOpenAIChatCompletion(
        modelId: "Phi-4-Reasoning",
        endpoint: new Uri("http://localhost:1234/v1/"),
        apiKey: "LMStudio")
    .Build();

Installed Packages Microsoft.Extensions.Configuration, 9.0.6 Microsoft.Extensions.Configuration.Json, 9.0.6 Microsoft.SemanticKernel, 1.60.0